# Fold-wise nnU-Net prediction and structure-level evaluation

Configure `BASE_DIR` and `fold` for your environment before running. The `fold` variable is used to select the validation split and to name fold-specific ground-truth and prediction directories. This notebook retains the core prediction and metric workflow; directory listing and one-off debugging cells have been removed.

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/path/to/nnunet")
DATASET_ID = "Dataset002_SPIDER"
n = 0  # Set to the desired n index, e.g. 0, 1, 2, 3, or 4

RAW_DIR = BASE_DIR / "nnUNet_raw" / DATASET_ID
PREPROCESSED_DIR = BASE_DIR / "nnUNet_preprocessed" / DATASET_ID
RESULTS_DIR = BASE_DIR / "nnUNet_results" / DATASET_ID

GROUND_TRUTH_DIR = RAW_DIR / f"groundtruth_fold{n}"
PREDICTIONS_DIR = RESULTS_DIR / f"predictions_fold{n}"
IMAGES_TS_DIR = GROUND_TRUTH_DIR / "imagesTs"
LABELS_TS_DIR = GROUND_TRUTH_DIR / "labelsTs"

# nnU-Net v2 environment variables
os.environ["nnUNet_raw"] = str(BASE_DIR / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(BASE_DIR / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(BASE_DIR / "nnUNet_results")


In [ ]:
import shutil

imagesTr = RAW_DIR / "imagesTr"
labelsTr = RAW_DIR / "labelsTr"

IMAGES_TS_DIR.mkdir(parents=True, exist_ok=True)
LABELS_TS_DIR.mkdir(parents=True, exist_ok=True)

for case in val_cases:
    shutil.copy2(imagesTr / f"{case}_0000.nii.gz", IMAGES_TS_DIR / f"{case}_0000.nii.gz")
    shutil.copy2(labelsTr / f"{case}.nii.gz", LABELS_TS_DIR / f"{case}.nii.gz")


In [ ]:
import os
import subprocess

env = os.environ.copy()
env["nnUNet_n_proc_DA"] = "0"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"

command = [
    "nnUNetv2_predict",
    "-i", str(IMAGES_TS_DIR),
    "-o", str(PREDICTIONS_DIR),
    "-d", "2",
    "-c", "3d_fullres",
    "-f", str(n),
    "-chk", "checkpoint_best.pth",
]

subprocess.run(command, check=True, env=env)


In [ ]:
import os
import numpy as np
import nibabel as nib

gt_dir = str(LABELS_TS_DIR)
pred_dir = str(PREDICTIONS_DIR)

def dice_coefficient(gt, pred):
    gt = gt > 0
    pred = pred > 0
    intersection = np.logical_and(gt, pred).sum()
    return (2.0 * intersection) / (gt.sum() + pred.sum() + 1e-8)

dice_scores = []

for fname in sorted(os.listdir(gt_dir)):
    if not fname.endswith(".nii.gz"):
        continue

    gt = nib.load(os.path.join(gt_dir, fname)).get_fdata()
    pred = nib.load(os.path.join(pred_dir, fname)).get_fdata()

    d = dice_coefficient(gt, pred)
    dice_scores.append(d)

    print(f"{fname}: Dice = {d:.4f}")

print("\n✅ Mean Dice:", np.mean(dice_scores))
print("✅ Std Dice:", np.std(dice_scores))

In [ ]:
import SimpleITK as sitk
import numpy as np
from medpy.metric import binary

import os

gt_dir = str(LABELS_TS_DIR)
pred_dir = str(PREDICTIONS_DIR)

labels = [1,2,3,4,5,6,7,8,9,10,11]  # change if multi-class (e.g., [1,2,3])

results = {}

for case in sorted(os.listdir(gt_dir)):
    if not case.endswith(".nii.gz"):
        continue

    gt = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(gt_dir, case)))
    pred = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(pred_dir, case)))

    results[case] = {}

    for lbl in labels:
        gt_bin = (gt == lbl)
        pred_bin = (pred == lbl)

        if gt_bin.sum() == 0:
            continue

        dice = binary.dc(pred_bin, gt_bin)
        hd95 = binary.hd95(pred_bin, gt_bin)
        asd = binary.asd(pred_bin, gt_bin)

        results[case][f"class_{lbl}"] = {
            "Dice": dice,
            "HD95": hd95,
            "ASD": asd
        }

results

In [ ]:
#HD95 and ASD for vertebrae and IVDs seperately
import numpy as np
from medpy.metric import binary
# Define label IDs
VERTEBRA_LABELS = [1, 2, 3, 4, 5]          # adjust if needed
IVD_LABELS      = [6, 7, 8, 9, 10, 11]     # adjust if needed
# Vertebra masks
gt_vert   = np.isin(gt, VERTEBRA_LABELS)
pred_vert = np.isin(pred, VERTEBRA_LABELS)

# IVD masks
gt_ivd   = np.isin(gt, IVD_LABELS)
pred_ivd = np.isin(pred, IVD_LABELS)

#Compute HD95 and ASD per structure
#Vertebrae
if gt_vert.any() and pred_vert.any():
    dice_vert = binary.dc(pred_vert, gt_vert)
    hd95_vert = binary.hd95(pred_vert, gt_vert)
    asd_vert  = binary.asd(pred_vert, gt_vert)
else:
    dice_vert = np.nan
    hd95_vert = np.nan
    asd_vert  = np.nan

print("Vertebrae Dice:", dice_vert)
print("Vertebrae HD95:", hd95_vert)
print("Vertebrae ASD :", asd_vert)

#IVD
if gt_ivd.any() and pred_ivd.any():
    dice_ivd = binary.dc(pred_ivd, gt_ivd)
    hd95_ivd = binary.hd95(pred_ivd, gt_ivd)
    asd_ivd  = binary.asd(pred_ivd, gt_ivd)
else:
    dice_ivd = np.nan
    hd95_ivd = np.nan
    asd_ivd  = np.nan

print("IVD Dice:", dice_ivd)
print("IVD HD95:", hd95_ivd)
print("IVD ASD :", asd_ivd)

In [ ]:
import numpy as np
import pandas as pd

num_classes = 11
table_rows = []

for idx, (case, class_dict) in enumerate(sorted(results.items()), start=1):
    row = {
        "Number": idx,
        "Case": case
    }
    
    # Initialize all class columns with NaN
    for c in range(1, num_classes + 1):
        row[f"Class_{c}"] = np.nan
    
    # Fill Dice values
    for cls_name, metrics in class_dict.items():
        cls_id = int(cls_name.replace("class_", ""))
        row[f"Class_{cls_id}"] = metrics["Dice"]
    
    table_rows.append(row)

df = pd.DataFrame(table_rows)
df

In [ ]:
import numpy as np

class_cols = [f"Class_{i}" for i in range(1, 12)]

mean_dice_per_class = df[class_cols].mean()
overall_mean_dice = mean_dice_per_class.mean()
overall_std_dice = mean_dice_per_class.std()

print(mean_dice_per_class)
print(f"\nOverall Mean Dice = {overall_mean_dice:.3f} ± {overall_std_dice:.3f}")

In [ ]:
# Create / assign DataFrame
df_file = df.copy()

# Save to CSV
df_file.to_csv(f"dice_fold{n}.csv", index=False)